In [4]:
!pip install faiss-cpu sentence-transformers -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 59.5 MB/s eta 0:00:00


In [8]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import pipeline

# Passo 1: Inicialização do modelo de embeddings
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Passo 2: Preparação do Corpus de Conhecimento
documents = [
    "RAG (Retrieval-Augmented Generation) é uma técnica que combina recuperação de dados com modelos de linguagem.",
    "O funcionamento do RAG envolve buscar documentos relevantes em um banco vetorial antes de gerar a resposta.",
    "FAISS é uma biblioteca para busca eficiente de similaridade em vetores densos.",
    "Modelos Seq2Seq como o T5 podem ser usados para gerar texto baseado em um contexto fornecido."
]

# Conversão dos documentos em embeddings
doc_embeddings = embedder.encode(documents)

# Passo 3: Criação do Índice FAISS
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(doc_embeddings).astype('float32'))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

# Carregando modelo e tokenizer explicitamente para evitar erros de task registry
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def rag_query(query, k=2):
    # Converte consulta em vetor
    query_vector = embedder.encode([query])

    # Busca os top-k documentos mais similares
    distances, indices = index.search(np.array(query_vector).astype('float32'), k)

    # Recupera o texto dos documentos
    context = " ".join([documents[i] for i in indices[0]])

    # Geração com o modelo Seq2Seq
    prompt = f"Pergunta: {query}\n\nContexto: {context}\n\nResposta baseada no contexto:"

    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(**inputs, max_length=150)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return response, [documents[i] for i in indices[0]]

# Exemplo de execução
pergunta = "O que é o RAG e como ele funciona?"
resposta, fontes = rag_query(pergunta)


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [10]:
print(f"Resposta: {resposta}\nFontes: {fontes}")

Resposta: Seq2Seq
Fontes: ['Modelos Seq2Seq como o T5 podem ser usados para gerar texto baseado em um contexto fornecido.', 'FAISS é uma biblioteca para busca eficiente de similaridade em vetores densos.']
